# IPC Protocol (020)
Demonstrate the length-prefixed JSON protocol and request handling.

In [1]:
%load_ext autoreload
%autoreload 2
import os
import tempfile
from pathlib import Path

from ciphercache.daemon.state import DaemonConfig, DaemonState
from ciphercache.ipc.handler import handle_request
from ciphercache.ipc.framing import decode_single_frame, encode_message
from ciphercache.ttl import parse_ttl

In [2]:
data_dir = Path(tempfile.mkdtemp(prefix="ciphercache-demo-"))
os.chmod(data_dir, 0o700)
config = DaemonConfig(data_dir=data_dir)
state = DaemonState(config=config)
state.unlock(parse_ttl("1h"))
state.secrets = {"service/api": {"api_key": "demo"}}
ticket_path = state.issue_ticket("demo_client")
print(ticket_path)
ticket = ticket_path.read_text(encoding="utf-8").strip()
print(ticket)


/var/folders/0t/w9l_5c597rdglh2kbffq18sr0000gn/T/ciphercache-demo-bhf6esrz/tickets/demo_client.ticket
w-z2fAh7EdFrpLPqKxdrv-a-DSC3GsmvaTr2xjWFok8


In [3]:
ttl_examples = ["5s", "1h 30m", "infinity"]
{value: parse_ttl(value) for value in ttl_examples}


{'5s': 5, '1h 30m': 5400, 'infinity': None}

In [4]:
message = {"version": "v0", "id": "frame-1", "type": "request", "op": "ping", "payload": {}}
frame = encode_message(message)
decoded = decode_single_frame(frame)
{"frame_len": len(frame), "roundtrip_ok": decoded == message, "decoded": decoded}


{'frame_len': 77,
 'roundtrip_ok': True,
 'decoded': {'version': 'v0',
  'id': 'frame-1',
  'type': 'request',
  'op': 'ping',
  'payload': {}}}

In [ ]:
request = {
    "version": "v0",
    "id": "demo-1",
    "type": "request",
    "op": "get_secret",
    "payload": {
        "ticket": ticket,
        "secret_name": "service/api",
    },
}
handle_request(state, request)